In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import DATA_DIR, FEWSHOT_CACHE_DIR, PROMPT_DIR, GREEDY_CONFIG, PROFILE_CACHE_DIR
import json
from src.extractor import LabelTransformConfig, prepare_label_tokens, _parse_parent_annotations
from src.rpr import ReferenceProfileRegistry
from src.ann_extractor import extract_parent_level_annotations
from datetime import datetime
from src.htmlLabel import ReferenceMention
from src.tokenizer_utils import tokenize, decode
from src.htmlLabel import simplified_to_normal_form
from src.models import get_messages
from tqdm import tqdm
import random



c:\Users\zakga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Choose the prompting configuration

In [2]:
filename = "1989CanLII1415ONCA"
split = "test"
filepath = Path(DATA_DIR) / "annotated" / split / f"{filename}.html"
#filepath = Path("output")  / f"{filename}_0.html"

#filepath = PROJECT_ROOT / Path("output") / "dev_gpt5.2_DEC_fs6_greedy_sentence_long" / filename / f"{filename}_2.html"
with open(filepath, "r", encoding="utf-8") as f:
    html_content = f.read()

gold_mentions = extract_parent_level_annotations(html_content)


#### Common Few SHot Selection

In [3]:
fewshot_method = "random"   # "greedy" | "random"

fewshot_filename = f"examples_coref_{fewshot_method}"

with open(FEWSHOT_CACHE_DIR / f"{fewshot_filename}.json", "r", encoding="utf-8") as f:
    fewshot_file_content = json.load(f)

fewshot_examples = [
    (
        json.loads(example["input"]),
        example["output"],
        example["meta"],
    )
    for example in fewshot_file_content["examples"]
]
print("fewshot examples from :", fewshot_filename)


fewshot examples from : examples_coref_random


##### Few Shot processing step

In [4]:
def sample_reference_profile_subset(
    rpr: ReferenceProfileRegistry,
    docid,
    doc_type: str = None,
    include_docid="yes",
    length: int = None,
    min_length: int = None,
    max_length: int = None,
    seed: int = None,
    p: float = 0.7,
    must_include_main_title: bool = True
) -> ReferenceProfileRegistry:
    """
    Build a random subset of `rpr` as a new ReferenceProfileRegistry.

    Parameters
    ----------
    rpr : ReferenceProfileRegistry
        The full list of profiles to sample from.
    docid :
        The docid we care about when deciding inclusion.
    doc_type : str, optional
        If given, only profiles with this `doc_type` are considered for the subset.
    include_docid : {"yes", "no", "random"}
        - "yes":    the profile with `docid` is forced into the subset.
        - "no":     the profile with `docid` is forced OUT of the subset.
        - "random": the profile with `docid` is included with probability `p`.
    length : int, optional
        Exact size of the returned subset. If given, takes priority over
        min_length/max_length.
    min_length, max_length : int, optional
        If `length` is not given, the subset size is drawn uniformly from
        [min_length, max_length] (inclusive), using `seed`.
    seed : int, optional
        Seed for all the random choices made in this function (size choice,
        whether to include the target docid in "random" mode, and which
        other profiles fill the rest of the subset). Uses a local
        random.Random instance, so global random state is untouched.
    p : float, default 0.7
        Probability of including the target docid's profile when
        include_docid="random". Ignored otherwise.

    Returns
    -------
    ReferenceProfileRegistry
        A new list containing the sampled subset of profiles.

    Raises
    ------
    ValueError
        If include_docid is not one of "yes"/"no"/"random"; if include_docid
        is "yes" but no profile with `docid` exists in `rpr`; if neither
        `length` nor a valid (min_length, max_length) pair is given; or if
        the requested subset size is larger than what's available.
    """
    if include_docid not in ("yes", "no", "random"):
        raise ValueError(
            f"include_docid must be 'yes', 'no', or 'random', got {include_docid!r}"
        )

    rng = random.Random(seed)

    all_profiles = list(rpr)
    if doc_type is not None:
        all_profiles = [prof for prof in all_profiles if prof.doc_type == doc_type]
    if must_include_main_title:
        all_profiles = [prof for prof in all_profiles if prof.main_title is not None]
    target_profile = rpr.get_profile_by_docid(docid)

    if include_docid == "yes" and target_profile is None:
        raise ValueError(f"docid {docid!r} not found in the given ReferenceProfileRegistry")

    # Decide, for this call, whether the target profile should be forced in,
    # forced out, or absent because it doesn't exist.
    force_include_target = False
    force_exclude_target = False

    if target_profile is None:
        # Nothing to force either way; "no" and "random" are trivially satisfied.
        force_exclude_target = True
    elif include_docid == "yes":
        force_include_target = True
    elif include_docid == "no":
        force_exclude_target = True
    else:  # "random"
        if rng.random() < p:
            force_include_target = True
        else:
            force_exclude_target = True

    # Pool of profiles eligible to fill the "free" slots of the subset
    # (everything except the target profile, which is handled separately).
    other_profiles = [prof for prof in all_profiles if prof is not target_profile]

    # Work out the desired subset size.
    # Max possible size of the final subset given the forced inclusion/exclusion:
    if force_include_target:
        max_possible = 1 + len(other_profiles)
    else:
        max_possible = len(other_profiles)

    if length is not None:
        subset_size = length
    else:
        if min_length is None or max_length is None:
            raise ValueError(
                "Either `length`, or both `min_length` and `max_length`, must be provided"
            )
        if min_length > max_length:
            raise ValueError("min_length cannot be greater than max_length")
        subset_size = rng.randint(min_length, max_length)

    if subset_size < 0:
        raise ValueError("Computed subset size is negative")

    # How many additional (non-target) profiles do we need to fill the subset?
    remaining_slots = subset_size - 1 if force_include_target else subset_size
    remaining_slots = max(remaining_slots, 0)

    chosen_others = rng.sample(other_profiles, min(remaining_slots, len(other_profiles))) if remaining_slots > 0 else []

    subset_profiles = list(chosen_others)
    if force_include_target:
        subset_profiles.append(target_profile)

    # Shuffle so the target profile (if forced in) isn't always last.
    rng.shuffle(subset_profiles)

    result = ReferenceProfileRegistry()
    for prof in subset_profiles:
        result.add_profile(prof)

    return result

In [5]:
def format_profile_for_prompt(profile_dict: dict, max_fragments: int = 10) -> dict:
    """
    Take one profile's to_dict() output (with tracked fields still in
    {value: first_seen_id} form) and turn it into a prompt-friendly dict:
    - tracked fields become plain lists of their keys (ids dropped)
    - empty lists are omitted entirely
    - fragments_mentioned is truncated to the last `max_fragments` items
    """
    tracked_fields = ("alternative_titles", "citations", "fragments_mentioned", "authors")

    formatted = {}
    for key, value in profile_dict.items():
        if key in tracked_fields:
            values_list = list(value.keys()) if isinstance(value, dict) else list(value)
            if key == "fragments_mentioned":
                values_list = values_list[-max_fragments:]
            if not values_list:
                continue  # drop empty lists
            formatted[key] = values_list
        else:
            formatted[key] = value

    return formatted



def example_to_string(example_input: dict, docid: str, doctype: str, max_fragments: int = 10) -> str:
    """
    Given one fewshot example's `input` dict (with keys "input_mention",
    "context", "profileRegistry"), build a single formatted string
    describing the input. Output mention is intentionally not included.
    """
    rpr = ReferenceProfileRegistry.from_dict(example_input["profileRegistry"])

    attributes = ["main_title", "alternative_titles",
                  "citations", "fragments_mentioned", "authors"]

    
    filtered_rpr = sample_reference_profile_subset(
        rpr=rpr,
        docid=docid,
        include_docid = "yes",
        doc_type=doctype,
        min_length = 1,
        max_length = 10,
        seed=None,
        p = 0.8,
    )
    

    rpr_main_title = filtered_rpr.replace_docid_with_main_title()
        
    profiles_formatted = [
        format_profile_for_prompt(profile.to_dict(attributes=attributes), max_fragments=max_fragments)
        for profile in rpr_main_title
    ]

    lines = []
    lines.append(f"Input mention: {example_input['input_mention']}")
    lines.append(f"Context: {example_input['context']}")
    lines.append("Reference Profile Registry:")
    for i, profile in enumerate(profiles_formatted):
        lines.append(f"  Profile {i}: {profile}")

    return "\n".join(lines)

In [6]:
seed = 42
nb_fewshot_examples = 6
from src.rpr import ReferenceProfileRegistry

rng = random.Random(seed)
nb_fewshot_examples = 6

final_fewshot = []
for i in range(6):
    example = fewshot_examples[i]
    input_, output, meta = example
    rpr = ReferenceProfileRegistry.from_dict(input_["profileRegistry"])

    final_input = example_to_string(input_, docid=meta["docid"], doctype=input_["input_mention"].split(">")[0][1:])
    final_fewshot.append((final_input, output))


#### Common Prompt loading

In [7]:
prompt_filename = "coref_long.txt"

with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)

system_prompt used :  coref_long.txt


#### Assistant loading

In [8]:
MODEL_MAPPING_NAME = {
    "qwen7b": "Qwen2.5-7B-Instruct",
    "qwen32b": "Qwen2.5-32B-Instruct",
    "gpt-5.2": "gpt-5.2",
    "phi-4": "phi-4",
    "saul-54b": "SaulLM-54B-Instruct"
}

from src.models import AssistantFactory

model = "gpt-5.2"

if model == "gpt-5.2":
        assistant = AssistantFactory.create_from_config({
            "type": "openai",
            "model_name": model,
            "temperature": 1,
        })
else:
    assistant = AssistantFactory.create(MODEL_MAPPING_NAME[model])


In [9]:
rpr = ReferenceProfileRegistry()

In [10]:
config = LabelTransformConfig(
    use_simplified=True,
    switch_type=False,
    keep_attributes=["labelname"]
) # We only remove the attribute
segment = segments[1]
mention = segment["tokens"]
html_label = segment["meta"]["label"]


prepared_tokens = prepare_label_tokens(mention, config)
user_input = decode(prepared_tokens)

NameError: name 'segments' is not defined

In [ ]:
user_input

'<decision><title>Housen <i>v</i>. Nikolaisen</title>, <citation>[2002] 2 S.C.R. 235</citation>, <citation>2002 SCC 33</citation></decision>'

In [ ]:
filtered_fewshot = []
messages = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=final_fewshot, has_system_role=assistant.has_system_role, prefix="Annotate this mention: ")

In [ ]:
messages[-1]

{'role': 'user',
 'content': 'Annotate this mention: <decision><title>Housen <i>v</i>. Nikolaisen</title>, <citation>[2002] 2 S.C.R. 235</citation>, <citation>2002 SCC 33</citation></decision>'}

In [ ]:
generated = assistant.generate(messages=messages)

In [ ]:
print(generated)

<decision docid="NEW_REFERENCE"><title>Housen <i>v</i>. Nikolaisen</title>, <citation>[2002] 2 S.C.R. 235</citation>, <citation>2002 SCC 33</citation></decision>


In [ ]:
from src.htmlLabel import ReferenceMention
mention_generated = ReferenceMention(generated)

In [ ]:
print(mention_generated.sublabels)
print(mention_generated.html_str)
print(mention_generated.get_sublabel_texts()["title"][0])

['title', 'citation', 'citation']
<manual_label docid="NEW_REFERENCE" labelname="decision"><manual_label labelname="title">Housen <i>v</i>. Nikolaisen</manual_label>, <manual_label labelname="citation">[2002] 2 S.C.R. 235</manual_label>, <manual_label labelname="citation">2002 SCC 33</manual_label></manual_label>
Housenv. Nikolaisen


In [ ]:
rpr.update_from_mention(mention_generated)

{'doc_type': 'decision', 'jurisdiction': None, 'main_title': 'Housenv. Nikolaisen', 'docid': 'Housenv. Nikolaisen', 'alternative_titles': {}, 'citations': {'[2002] 2 S.C.R. 235': None, '2002 SCC 33': None}, 'fragments_mentioned': {}, 'authors': {}, 'first_seen_id': None}

In [ ]:
print(rpr)

{'profiles': [{'doc_type': 'decision', 'jurisdiction': None, 'main_title': 'Housenv. Nikolaisen', 'docid': 'Housenv. Nikolaisen', 'alternative_titles': {}, 'citations': {'[2002] 2 S.C.R. 235': None, '2002 SCC 33': None}, 'fragments_mentioned': {}, 'authors': {}, 'first_seen_id': None}]}


In [9]:
import re
from typing import Optional

# Matches docid="value" or docid=\"value\" (escaped quotes), non-greedy,
# stops at the first closing quote.
_DOCID_RE = re.compile(r'docid\s*=\s*\\?["\'](.*?)\\?["\']', re.DOTALL)


def extract_docid_from_generation(generated: str) -> Optional[str]:
    """
    Extract the value of the first `docid="..."` attribute found in the
    LLM-generated string, tolerating malformed/unclosed markup.

    Returns the stripped docid value, or None if not found / empty.
    """
    if not generated:
        return None

    match = _DOCID_RE.search(generated)
    if not match:
        return None

    docid = match.group(1).strip()
    return docid or None

In [10]:
rpr = ReferenceProfileRegistry()

attributes = ["main_title", "alternative_titles",
                  "citations", "fragments_mentioned", "authors"]

config = LabelTransformConfig(
    use_simplified=True,
    switch_type=False,
    keep_attributes=["labelname"]
)  # We only remove the attribute

max_fragments = 10

mentions = extract_parent_level_annotations(html_content)
result = {}

failed_count = 0
log_path = "processing_log_1.txt"

with open(log_path, "a", encoding="utf-8") as log_file:
    
    for idx, mention in tqdm(enumerate(mentions), total=len(mentions)):
        id = mention.html_tag.attributes["id"]
        prepared_tokens = prepare_label_tokens(tokenize(mention.html_str), config)

        profiles_formatted = [
                format_profile_for_prompt(profile.to_dict(attributes=attributes), max_fragments=max_fragments)
                for profile in rpr
            ]

        lines = []
        lines.append("Reference Profile Registry:")
        for i, profile in enumerate(profiles_formatted):
            lines.append(f"  Profile {i}: {profile}")

        profiles_str ="\n".join(lines)


        user_input = decode(prepared_tokens) + "\n" + profiles_str

        messages = get_messages(
            system_prompt=system_prompt,
            user_input=user_input,
            fewshot_examples=final_fewshot,
            has_system_role=assistant.has_system_role,
            prefix="Annotate this mention: ",
        )

        generated = assistant.generate(messages=messages)

        docid_generated = extract_docid_from_generation(generated)
        if docid_generated is None:
            failed_count += 1
            print(f"Failed to extract docid for mention {idx} (id={id})")
            continue
        #print(f"generated : {generated} | docid extracted : {docid_generated}")

        

        mention.html_tag.set_attribute("docid", docid_generated)

        profile = rpr.update_from_mention(ReferenceMention(mention.html_str))
            

        result[id] = docid_generated



                # ---- logging ----
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "id": id,
            "user_input": decode(prepared_tokens),
            "docid_generated": docid_generated,
            "main_title": profile.main_title,
            "profile_docid": profile.docid,
        }

        log_file.write(f"{'='*80}\n")
        log_file.write(f"MENTION #{idx}\n")
        log_file.write(f"{'='*80}\n")
        log_file.write(json.dumps(log_entry, indent=2, default=str, ensure_ascii=False))
        log_file.write("\n\n")
        log_file.flush()


print(f"Failed: {failed_count} / {len(mentions)}")

100%|██████████| 95/95 [02:20<00:00,  1.48s/it]

Failed: 0 / 95


In [11]:
from collections import defaultdict
def dict_to_clusters(mapping):
    """
    Parameters
    ----------
    mapping : dict[int, str]
        mention_id -> predicted canonical title

    Returns
    -------
    list[list[int]]
    """

    clusters = defaultdict(list)

    for mention_id, entity in mapping.items():
        clusters[entity].append(mention_id)

    return list(clusters.values())

In [12]:
system_clusters = dict_to_clusters(result)
with open("../output/system_clusters_1989.json", "w", encoding="utf-8") as f:
    json.dump(system_clusters, f, indent=2, ensure_ascii=False)

In [13]:
ground_truth = {mention.html_tag.attributes["id"]: mention.html_tag.attributes.get("docid") for mention in gold_mentions}

In [14]:
cluster_ground_truth = dict_to_clusters(ground_truth)
with open("../output/ground_truth_clusters_1989.json", "w", encoding="utf-8") as f:
    json.dump(cluster_ground_truth, f, indent=2, ensure_ascii=False)

#### Evaluation

CaNLL shared tasks (2011-2012) standardized three metrics : 
MUC (Vilain et al. 1995) : Measures how many links must be added/deleted to transform one clustering into another.
B³ (Bagga & Baldwin)
CEAF : Finds the optimal one-to-one alignment between predicted and gold clusters.
LEA : Weights entities according to their importance.
The famous CoNLL score is simply

(MUC + B³ + CEAF)/3


In [ ]:
from collections import defaultdict


def dict_to_clusters(mapping):
    """
    Parameters
    ----------
    mapping : dict[int, str]
        mention_id -> predicted canonical title

    Returns
    -------
    list[list[int]]
    """

    clusters = defaultdict(list)

    for mention_id, entity in mapping.items():
        clusters[entity].append(mention_id)

    return list(clusters.values())



ModuleNotFoundError: No module named 'coval.eval'

In [ ]:
scores = evaluate_coreference(ground_truth, result)

for metric, value in scores.items():
    print(f"{metric:10s}: {100*value:.2f}")

NameError: name 'evaluate_coreference' is not defined